## 1. Inspect the source workbook

This cell checks the available sheets, their size, and the column names so we know what data we
have before making changes.

In [4]:
%pip install pandas openpyxl numpy matplotlib seaborn scikit-learn streamlit

  Using cached matplotlib-3.11.2-cp313-cp313-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.4.0-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.1-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.27.1-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached python_multipart-0.0.32-py3-none-


[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from pathlib import Path
import pandas as pd

file_path = Path("data/raw/online_retail_II.xlsx")
workbook = pd.ExcelFile(file_path)
for sheet_name in workbook.sheet_names:
    sample = pd.read_excel(workbook, sheet_name=sheet_name, nrows=5)
    print(f"\nSheet: {sheet_name}")
    print("Columns:", sample.columns.tolist())
    display(sample)


Sheet: Year 2009-2010
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom



Sheet: Year 2010-2011
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [5]:
frames = [
    pd.read_excel(workbook, sheet_name=sheet_name)
    for sheet_name in workbook.sheet_names
]

transactions = pd.concat(frames, ignore_index=True)
transactions["InvoiceDate"] = pd.to_datetime(
    transactions["InvoiceDate"], errors="coerce"
)

print("Rows:", len(transactions))
print("Date range:", transactions["InvoiceDate"].min(),
      "to", transactions["InvoiceDate"].max())
print("Missing customer IDs:", transactions["Customer ID"].isna().sum())
print("Missing dates:", transactions["InvoiceDate"].isna().sum())
print("Invoices starting with C:",
      transactions["Invoice"].astype(str).str.upper().str.startswith("C").sum())
print("Quantity <= 0:", (transactions["Quantity"] <= 0).sum())
print("Price <= 0:", (transactions["Price"] <= 0).sum())
print("Exact duplicate rows:", transactions.duplicated().sum())

Rows: 1067371
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Missing customer IDs: 243007
Missing dates: 0
Invoices starting with C: 19494
Quantity <= 0: 22950
Price <= 0: 6207
Exact duplicate rows: 34335


## 3. Define valid purchases

Retention requires an identifiable customer and a completed purchase.
This cell removes rows without a customer ID, cancellation invoices,
and rows with nonpositive quantity or price. It keeps the original
data unchanged and reports how many rows remain.

In [6]:
valid_purchases = transactions.loc[
    transactions["Customer ID"].notna()
    & ~transactions["Invoice"].astype(str).str.upper().str.startswith("C")
    & (transactions["Quantity"] > 0)
    & (transactions["Price"] > 0)
    & transactions["InvoiceDate"].notna()
].copy()

valid_purchases["line_revenue"] = (
    valid_purchases["Quantity"] * valid_purchases["Price"]
)

print("Original rows:", len(transactions))
print("Valid purchase rows:", len(valid_purchases))
print("Rows excluded:", len(transactions) - len(valid_purchases))
print("Customers:", valid_purchases["Customer ID"].nunique())
print("Invoices:", valid_purchases["Invoice"].nunique())
print("Valid-purchase date range:",
      valid_purchases["InvoiceDate"].min(),
      "to", valid_purchases["InvoiceDate"].max())

Original rows: 1067371
Valid purchase rows: 805549
Rows excluded: 261822
Customers: 5878
Invoices: 36969
Valid-purchase date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00


## 4. Count purchases per customer

Each invoice represents an order, while each row represents an item in that
order. This cell counts distinct invoices per customer to see whether the
dataset contains enough repeat purchases for a retention project.

In [7]:
orders_per_customer = (
    valid_purchases.groupby("Customer ID")["Invoice"]
    .nunique()
)

print("Customers:", len(orders_per_customer))
print("Customers with 1 order:", (orders_per_customer == 1).sum())
print("Customers with 2+ orders:", (orders_per_customer >= 2).sum())
print("Repeat-order share:",
      f"{(orders_per_customer >= 2).mean():.1%}")

display(orders_per_customer.describe())

Customers: 5878
Customers with 1 order: 1623
Customers with 2+ orders: 4255
Repeat-order share: 72.4%


count    5878.000000
mean        6.289384
std        13.009406
min         1.000000
25%         1.000000
50%         3.000000
75%         7.000000
max       398.000000
Name: Invoice, dtype: float64

## 5. Create an order-level table

The source has one row per product line. This cell groups lines from the
same customer and invoice into one order, with its date, item count, and
total value. We need this table to study purchase timing.

In [9]:
orders = (
    valid_purchases
    .groupby(["Customer ID", "Invoice"], as_index=False)
    .agg(
        order_date=("InvoiceDate", "min"),
        items_purchased=("Quantity", "sum"),
        order_value=("line_revenue", "sum"),
    )
    .sort_values(["Customer ID", "order_date"])
)

print("Order rows:", len(orders))
print("Customers:", orders["Customer ID"].nunique())
display(orders.head(10))

Order rows: 36969
Customers: 5878


,Customer ID,Invoice,order_date,items_purchased,order_value
0,12346.0,491725,2009-12-14 08:34:00,10,45.00
1,12346.0,491742,2009-12-14 11:00:00,5,22.50
2,12346.0,491744,2009-12-14 11:02:00,5,22.50
3,12346.0,492718,2009-12-18 10:47:00,5,22.50
4,12346.0,492722,2009-12-18 10:55:00,1,1.00
5,12346.0,493410,2010-01-04 09:24:00,5,22.50
6,12346.0,493412,2010-01-04 09:53:00,5,22.50
7,12346.0,494450,2010-01-14 13:50:00,5,22.50
8,12346.0,495295,2010-01-22 13:30:00,5,22.50
9,12346.0,499763,2010-03-02 13:08:00,5,27.05


## 6. Define a 90-day return outcome

As of June 1, 2011, include customers with an order in the previous
180 days. Label each customer 1 if they order in the next 90 days,
or 0 if they do not. This is one historical evaluation group.

In [10]:
cutoff = pd.Timestamp("2011-06-01")
history_start = cutoff - pd.Timedelta(days=180)
outcome_end = cutoff + pd.Timedelta(days=90)

past_orders = orders[
    (orders["order_date"] >= history_start)
    & (orders["order_date"] < cutoff)
]

future_orders = orders[
    (orders["order_date"] >= cutoff)
    & (orders["order_date"] < outcome_end)
]

eligible_customers = set(past_orders["Customer ID"])
returning_customers = set(future_orders["Customer ID"])

print("History window:", history_start, "to", cutoff)
print("Outcome window:", cutoff, "to", outcome_end)
print("Eligible customers:", len(eligible_customers))
print("Returned within 90 days:",
      len(eligible_customers & returning_customers))
print("90-day return share:",
      f"{len(eligible_customers & returning_customers) / len(eligible_customers):.1%}")

History window: 2010-12-03 00:00:00 to 2011-06-01 00:00:00
Outcome window: 2011-06-01 00:00:00 to 2011-08-30 00:00:00
Eligible customers: 2659
Returned within 90 days: 1331
90-day return share: 50.1%


## 7. Build customer features

For each eligible customer, calculate how recently they ordered, how many
orders they placed, and how much they spent in the 180-day history window.
The return label comes from the later 90-day window and is kept separate
from the features.

In [12]:
customer_features = (
    past_orders.groupby("Customer ID")
    .agg(
        last_order_date=("order_date", "max"),
        order_count_180d=("Invoice", "nunique"),
        spend_180d=("order_value", "sum"),
        average_order_value=("order_value", "mean"),
    )
)

customer_features["days_since_last_order"] = (
    cutoff - customer_features["last_order_date"]
).dt.days

customer_features["returned_90d"] = (
    customer_features.index.isin(returning_customers).astype(int)
)

customer_features = customer_features.reset_index()

print("Customer rows:", len(customer_features))
print("Return labels:")
print(customer_features["returned_90d"].value_counts())
display(customer_features.head())

Customer rows: 2659
Return labels:
returned_90d
1    1331
0    1328
Name: count, dtype: int64


,Customer ID,last_order_date,order_count_180d,spend_180d,average_order_value,days_since_last_order,returned_90d
0,12346.0,2011-01-18 10:01:00,1,77183.60,77183.600000,133,0
1,12347.0,2011-04-07 10:43:00,3,2535.22,845.073333,54,1
2,12348.0,2011-04-05 10:47:00,3,1487.24,495.746667,56,0
3,12350.0,2011-02-02 16:01:00,1,334.40,334.400000,118,0
4,12352.0,2011-03-22 16:08:00,5,1561.81,312.362000,70,0


## 8. Inspect unusually large orders

Large purchases may be valid, especially if some customers are wholesalers.
This cell checks the largest orders and the spread of customer spending
before we decide how to handle extreme values in a model.

In [14]:
print("Largest orders:")
display(
    orders.nlargest(10, "order_value")[
        ["Customer ID", "Invoice", "order_date", "order_value"]
    ]
)

print("Customer spend distribution:")
display(
    customer_features["spend_180d"].describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    )
)

Largest orders:


,Customer ID,Invoice,order_date,order_value
25633,16446.0,581483,2011-12-09 09:15:00,168469.60
11,12346.0,541431,2011-01-18 10:01:00,77183.60
11306,14156.0,493819,2010-01-07 12:34:00,44051.60
17833,15098.0,556444,2011-06-10 15:28:00,38970.00
31570,17450.0,524181,2010-09-27 16:59:00,33167.80
35979,18102.0,537659,2010-12-07 16:43:00,31770.98
31602,17450.0,567423,2011-09-20 11:05:00,31698.16
35970,18102.0,526934,2010-10-14 09:46:00,26007.08
35946,18102.0,515944,2010-07-15 15:29:00,22863.36
294,12415.0,556917,2011-06-15 13:37:00,22775.93


Customer spend distribution:


count      2659.000000
mean       1300.144385
std        4382.389656
min           2.900000
50%         511.360000
90%        2399.304000
95%        3762.143000
99%       12392.937800
max      110713.120000
Name: spend_180d, dtype: float64